# README: BERTopic Time-Series Pipeline (Finnhub)

## Purpose
Comprehensive BERTopic topic modeling pipeline with rigorous temporal validation and multi-seed robustness evaluation on financial news headlines from Finnhub.

## Input Data
- **Source**: `Data/finnhub_model_input.csv`
- **Required Columns**: `headline` (text), `date` (timestamp)
- **Processing**: Chronologically sorted, deduplicated headlines (exact matches across tickers removed)

## Pipeline Steps

### 1. Data Preparation & Temporal Splitting (Leakage-Aware)
- **Input**: Raw CSV with headlines and dates
- **Steps**:
  - Parse dates and sort chronologically
  - Remove rows with missing headline or date
  - Deduplicate exact headline matches (prevent ticker-based repeats from dominating topics)
  - Create three **temporal, non-overlapping splits** (70% train, 15% validation, 15% test) with day-level embargo to prevent information leakage:
    - **Train**: earliest 70% of unique calendar days
    - **Validation**: middle 15% of unique calendar days (kept separate for evaluation, not used in tuning decisions)
    - **Test**: latest 15% of unique calendar days (held out for final evaluation)
  - All documents on the same calendar day stay together in the same split

### 2. Embedding Generation
- **Model**: SentenceTransformer `all-MiniLM-L6-v2` (384-dim embeddings)
- **Device**: CUDA GPU if available, else CPU
- **Scope**: Generate embeddings only for training split (used for both initial fit and final fit during tuning)

### 3. Initial BERTopic Model Training (Full Train Split)
- **Architecture**:
  - UMAP for dimensionality reduction (15 neighbors, 10 components, cosine metric)
  - HDBSCAN for clustering (min_cluster_size=10, EOM selection method)
  - OnlineCountVectorizer for topic vocabulary (unigrams, custom stopwords including placeholder tokens)
  - Sentence-Transformers embeddings from step 2
- **Output**: Topic assignments for training documents, topic word lists

### 4. Validation Split Evaluation & Baseline Metrics
- **Action**: Project validation documents into trained topic space (no model retraining)
- **Metrics Computed** (train/val):
  - Semantic coherence: C_v and NPMI coherence
  - Topic diversity: Unique vocabulary fraction
  - Embedding-based metrics: Intra-topic similarity (within-cluster cosine), inter-topic similarity (between-cluster centroid cosine)
  - Silhouette score (robust variant excluding outliers and singletons)
  - Outlier ratio (fraction of documents assigned to label -1)

### 5. Metric Infrastructure & Coherence Computation
- **Define** helper functions for:
  - Text preprocessing (tokenization, n-gram support, stopword filtering)
  - Topic diversity calculation
  - Robust silhouette scoring (handles outliers, singleton clusters, edge cases)
  - Intra/inter-topic similarity in embedding space
  - Coherence scoring (C_v, NPMI via Gensim, with fallback and numerical stability controls)

### 6. Hyperparameter Tuning with Rolling Time-Series Cross-Validation
- **Search Space**: ~300 parameter combinations
  - `n_neighbors` (UMAP): [10, 30, 50, 70, 90]
  - `n_components` (UMAP): [4, 6, 8]
  - `min_cluster_size` (HDBSCAN): [8, 12, 16, 24, 32]
  - `min_samples` (HDBSCAN): [1, 2, 4, 6]
  - `ngram_range` (vectorizer): [(1, 2)]
- **Cross-Validation**: 3-fold rolling/expanding windows on train+validation pool (test remains untouched)
  - Each fold: Train on earlier days, validate on held-out later days
  - Fixed random seed (42) for deterministic model selection
- **Ranking Metric**: Equal-weighted composite score from 5 dimensions:
  - C_v coherence (higher better)
  - NPMI coherence (higher better)
  - Topic diversity (higher better)
  - Intra-topic similarity (higher better)
  - Inter-topic separation (lower inter-topic similarity is better, inverted during normalization)
  - All metrics min-max normalized to [0,1], then averaged
- **Best Model Selection**: Highest composite score configuration selected; fold-wise best configs also reported for temporal stability inspection

### 7. Final Model & Multi-Seed Test Evaluation
- **Training Data**: Refit best hyperparameters on combined train+validation split
- **Test Evaluation**: Apply final model to held-out test split
- **Robustness Analysis**: Re-fit and evaluate across 9 different random seeds [42, 7, 123, 2024, 99, 11, 77, 314, 2718]:
  - Each seed produces independent UMAP/HDBSCAN initialization
  - Compute all metrics on test split
  - Report mean ± std across seeds to quantify variability
- **Test Metrics** (reported mean ± std):
  - Coherence (C_v, NPMI)
  - Topic diversity
  - Intra/inter-topic similarity
  - Silhouette score
  - Outlier ratio
  - Topic count

### 8. Diagnostics & Visualization
- **Cluster Size Distribution**: Bar chart of topic sizes (excluding outliers)
- **2D Embedding Projection**: UMAP projection of test embeddings colored by point type (outlier/singleton/valid)
- **Seed-Level Variability**: Bar plots showing how outlier ratio and topic count vary across seeds

### 9. Export Results
- **Output Directory**: `Outputs/Finnhub_BERTopic/<YYYYMMDD_HHMMSS>/`
- **Exported Artifacts**:
  - `tables/`: Topic info, topic terms (long format)
  - `row_level/`: Row-wise topic assignments with probabilities
  - `models/`: BERTopic model artifact (for inference/further analysis)
  - `figures/`: HTML Plotly figures (if available)
  - `meta/`: JSON export summary with metadata
- **Reusability**: All outputs timestamped and organized for downstream thesis analysis and cross-model comparisons

## Key Design Decisions
- **Temporal Splitting**: Prevents look-ahead bias; respects chronological order of financial news
- **Fixed Seed for Tuning**: Ensures deterministic hyperparameter selection; randomness reserved for robustness analysis
- **Multi-Metric Ranking**: Avoids manual weighting; treats coherence, diversity, and geometry equally
- **Test Set Touch Rule**: Test data only used for final reporting (never in hyperparameter selection or early evaluation)
- **Robust Metrics**: Silhouette, outlier handling, and fallback mechanisms for edge cases (e.g., single valid cluster)

## Outputs & Next Steps
- Use exported topic assignments (`row_topic_assignments_train.csv`) to trace topics back to original documents
- Compare BERTopic results with LDA, Top2Vec baselines from parallel `03_*_pipeline_*.ipynb` notebooks
- Integrate into thesis tables and figures with results across datasets and seeds

In [1]:
import sys, torch
print('python_executable:', sys.executable)
print('torch_version:', torch.__version__)
print('torch_cuda_runtime:', torch.version.cuda)
print('cuda_available:', torch.cuda.is_available())
print('gpu_count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))

python_executable: c:\Users\gianf\AppData\Local\Programs\Python\Python313\python.exe
torch_version: 2.11.0+cu128
torch_cuda_runtime: 12.8
cuda_available: True
gpu_count: 1
gpu_name: NVIDIA GeForce RTX 3080


Imports

In [2]:
import pandas as pd
import numpy as np
import torch
from umap import UMAP
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load Data

In [3]:
df = pd.read_csv('Data/finnhub_model_input.csv')
print('Loaded: Data/finnhub_model_input.csv')
print('Shape:', df.shape)
display(df.head())

Loaded: Data/finnhub_model_input.csv
Shape: (4991, 5)


,stock,date,date_only,headline,headline_raw
0,ING,2024-09-17 15:00:55,2024-09-17,What are share buybacks?,What are share buybacks?
1,SMFG,2025-04-02 03:26:24,2025-04-02,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...
2,BACHY,2025-04-02 03:26:24,2025-04-02,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...
3,BACHY,2025-04-02 07:33:46,2025-04-02,"Across Chinese Fintech And Bank Stocks, See Wh...","Across Chinese Fintech And Bank Stocks, See Wh..."
4,SMFG,2025-04-02 12:41:01,2025-04-02,Japan's __COMPANY__ Plans Stablecoin Developme...,Japan's SMBC Plans Stablecoin Development With...


In [4]:
# 1. Parse date and prepare chronological data
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'headline']).sort_values('date').reset_index(drop=True)

# Deduplicate semantically identical headlines (topic modeling should not overcount copies across tickers)
before_dedup = len(df)
df = df.drop_duplicates(subset=['headline']).sort_values('date').reset_index(drop=True)
after_dedup = len(df)

# 2. Define split proportions and embargo
SPLIT_TRAIN = 0.70
SPLIT_VAL = 0.25
SPLIT_TEST = 0.05

# 3. Calculate date-based boundaries (keep same day together)
unique_dates = np.array(sorted(df['date'].dt.floor('D').unique()))
n_dates = len(unique_dates)

train_end_idx = int(SPLIT_TRAIN * n_dates)
val_end_idx = int((SPLIT_TRAIN + SPLIT_VAL) * n_dates)

# 4. Create masks with embargo
day_code, _ = pd.factorize(df['date'].dt.floor('D'), sort=True)
mask_train = day_code < train_end_idx
mask_val = (day_code >= (train_end_idx)) & (day_code < val_end_idx)
mask_test = day_code >= (val_end_idx)

# 5. Extract splits
train_df = df[mask_train].copy()
val_df = df[mask_val].copy()
test_df = df[mask_test].copy()

# Final lists for the model
train_docs, train_timestamps = train_df['headline'].tolist(), train_df['date'].tolist()
val_docs, val_timestamps = val_df['headline'].tolist(), val_df['date'].tolist()
test_docs, test_timestamps = test_df['headline'].tolist(), test_df['date'].tolist()

print(f'Deduplication: {before_dedup} -> {after_dedup} rows')
print(f"Train size: {len(train_docs)}")
print(f"Validation size: {len(val_docs)}")
print(f"Test size: {len(test_docs)}")
print('Date ranges:')
print(f"  Train: {train_df['date'].min()} -> {train_df['date'].max()}")
print(f"  Val:   {val_df['date'].min()} -> {val_df['date'].max()}")
print(f"  Test:  {test_df['date'].min()} -> {test_df['date'].max()}")

Deduplication: 4991 -> 4289 rows
Train size: 1963
Validation size: 1174
Test size: 1152
Date ranges:
  Train: 2024-09-17 15:00:55 -> 2025-12-11 23:21:43
  Val:   2025-12-12 04:00:00 -> 2026-03-10 23:14:51
  Test:  2026-03-11 07:10:44 -> 2026-03-28 19:00:00


Prepare Models

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Add placeholder tokens to stopwords so they do not dominate topics
placeholder_stopwords = {'__ticker__', '__company__', 'ticker', 'company'}
model_stopwords = sorted(set(ENGLISH_STOP_WORDS).union(placeholder_stopwords))

sentence_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
train_embeddings = sentence_model.encode(
    train_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
print(f'Embeddings computed on device: {device}')

Torch CUDA available: True
GPU: NVIDIA GeForce RTX 3080


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 760.88it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 31/31 [00:00<00:00, 31.27it/s]

Embeddings computed on device: cuda


Train BERTopic

In [6]:
print("Training BERTopic model on training split...\n")

umap_model = UMAP(n_neighbors=15, n_components=10, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = OnlineCountVectorizer(stop_words=model_stopwords)

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  calculate_probabilities=True,
  verbose=True
)

# Train BERTopic only on train split
topics_train, probs_train = topic_model.fit_transform(train_docs, embeddings=train_embeddings)

print("\nModel training complete!")

2026-04-23 19:14:13,484 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic model on training split...



2026-04-23 19:14:27,977 - BERTopic - Dimensionality - Completed ✓
2026-04-23 19:14:27,978 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-23 19:14:28,245 - BERTopic - Cluster - Completed ✓
2026-04-23 19:14:28,249 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-23 19:14:28,305 - BERTopic - Representation - Completed ✓



Model training complete!


Get Topic Information

In [7]:
# Get topic information
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,696,-1_bank_week_market_2025,"[bank, week, market, 2025, stock, group, says,...",[__TICKER__ Asset Management Inc. Announces Ri...
1,0,82,0_commentary_fund_q1_international,"[commentary, fund, q1, international, 2025, q2...",[Thornburg Better World International Fund Q1 ...
2,1,58,1_stablecoin_crypto_banks_banco,"[stablecoin, crypto, banks, banco, major, swif...",[__COMPANY__ becomes first major bank to launc...
3,2,57,2_dividend_portfolio_yield_10,"[dividend, portfolio, yield, 10, stocks, high,...",[My Dividend Stock Portfolio: New July Dividen...
4,3,55,3_china_gold_stimulus_pboc,"[china, gold, stimulus, pboc, trade, chinese, ...",[Oil Leads Commodity Rally After US-China Truc...
5,4,46,4_buyback_share_programme_voting,"[buyback, share, programme, voting, rights, sh...",[__COMPANY__ share buyback programme - Declara...
6,5,46,5_afternoon_update_sector_financial,"[afternoon, update, sector, financial, stocks,...",[Sector Update: Financial Stocks Rise Late Aft...
7,6,45,6_loan_energy_financing_debt,"[loan, energy, financing, debt, secures, billi...",['Banks Ready $38 Billion of Debt for Data Cen...
8,7,35,7_fed_inflation_cut_rate,"[fed, inflation, cut, rate, yields, treasury, ...",[Reuters Reports __COMPANY__ Expects Fed To Cu...
9,8,33,8_earnings_q3_revenues_q2,"[earnings, q3, revenues, q2, highlights, rise,...",[__COMPANY__ PLC __TICKER__ Q3 Earnings and Re...


Visualize Topics

In [8]:
topic_model.visualize_documents(train_docs, embeddings=train_embeddings)

In [9]:
fig = topic_model.visualize_heatmap()
fig.show()

In [10]:
hierarchical_topics = topic_model.hierarchical_topics(train_docs)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.show()

print("💡 This shows the hierarchical structure of topics")

100%|██████████| 50/50 [00:00<00:00, 543.23it/s]


💡 This shows the hierarchical structure of topics


In [11]:
import plotly.express as px

topics_over_time = topic_model.topics_over_time(
    train_docs,
    train_timestamps,
    nr_bins=20
 )

tot = topics_over_time.copy()
tot = tot[tot['Topic'] != -1].copy()
tot['Timestamp'] = pd.to_datetime(tot['Timestamp']).dt.to_period('M').dt.to_timestamp()

topic_labels = topic_model.get_topic_info()[['Topic', 'Name']].copy()
topic_labels = topic_labels[topic_labels['Topic'] != -1]
tot = tot.merge(topic_labels, on='Topic', how='left')
tot['TopicLabel'] = tot['Topic'].astype(str) + ': ' + tot['Name'].fillna('')

stacked = (
    tot.groupby(['Timestamp', 'TopicLabel'], as_index=False)['Frequency']
       .sum()
       .sort_values(['Timestamp', 'Frequency'], ascending=[True, False])
)

fig = px.bar(
    stacked,
    x='Timestamp',
    y='Frequency',
    color='TopicLabel',
    title='Topic Frequency Over Time (Stacked)',
    labels={'Timestamp': 'Month', 'Frequency': 'Document Count', 'TopicLabel': 'Topic'}
)
fig.update_layout(
    barmode='stack',
    xaxis_title='Month',
    yaxis_title='Document Count',
    legend_title='Topic',
    hovermode='x unified'
 )
fig.show()

13it [00:00, 46.40it/s]


Preparing Validation

In [12]:
bertopic_topics = [
    [word for word, _ in topic_model.get_topic(t)]
    for t in topic_model.get_topics().keys() if t != -1
    if topic_model.get_topic(t) is not None
 ]

# Project validation documents into trained topic space
val_topics, val_probs = topic_model.transform(val_docs)

print("Validation transform complete (test set remains untouched).")

Batches: 100%|██████████| 37/37 [00:00<00:00, 59.77it/s]
2026-04-23 19:14:33,220 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-23 19:14:40,128 - BERTopic - Dimensionality - Completed ✓
2026-04-23 19:14:40,129 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-23 19:14:40,195 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-23 19:14:40,370 - BERTopic - Probabilities - Completed ✓
2026-04-23 19:14:40,371 - BERTopic - Cluster - Completed ✓


Validation transform complete (test set remains untouched).


In [13]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import nltk
import warnings
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('stopwords', quiet=True)

# Numerical stability controls
COHERENCE_EPS = 1e-8
COHERENCE_MIN_TEXTS = 20
COHERENCE_MIN_VOCAB = 30
SIMILARITY_MAX_POINTS_PER_TOPIC = 300

# Build a tokenizer aligned with BERTopic vocabulary (supports unigrams + bigrams)
stop_words = list(model_stopwords) if 'model_stopwords' in globals() else list(stopwords.words('english'))
coherence_vectorizer = CountVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b\w\w+\b'
)
coherence_analyzer = coherence_vectorizer.build_analyzer()

def preprocess_documents(doc_list):
    tokenized = [coherence_analyzer(str(doc).lower()) for doc in doc_list]
    return [tokens for tokens in tokenized if len(tokens) >= 2]

processed_train_docs = preprocess_documents(train_docs)
processed_val_docs = preprocess_documents(val_docs)

# Dictionary from train split only
id2word = Dictionary(processed_train_docs)

def topic_diversity(topics):
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    return len(unique_words) / len(all_words) if len(all_words) > 0 else 0.0

def _filter_topics_for_dictionary(topics, dictionary, min_words_per_topic=3):
    vocab = dictionary.token2id
    filtered = []
    for topic in topics:
        cleaned = []
        for term in topic:
            token = str(term).strip().lower()
            if token in vocab:
                cleaned.append(token)
        cleaned = list(dict.fromkeys(cleaned))
        if len(cleaned) >= min_words_per_topic:
            filtered.append(cleaned)
    return filtered

def safe_silhouette_score(
    embeddings,
    labels,
    outlier_label=-1,
    metric='cosine',
    min_cluster_size_for_metric=2,
    min_points_for_metric=10,
    drop_singletons=True
):
    if embeddings is None or labels is None:
        return np.nan

    labels_arr = np.asarray(labels)
    embeddings_arr = np.asarray(embeddings)

    if embeddings_arr.ndim != 2 or labels_arr.ndim != 1:
        return np.nan
    if embeddings_arr.shape[0] != labels_arr.shape[0]:
        return np.nan

    finite_mask = np.isfinite(labels_arr.astype(float, copy=False))
    valid_mask = (labels_arr != outlier_label) & finite_mask
    if valid_mask.sum() < min_points_for_metric:
        return np.nan

    labels_valid = labels_arr[valid_mask]
    emb_valid = embeddings_arr[valid_mask]

    if drop_singletons:
        unique_labels, counts = np.unique(labels_valid, return_counts=True)
        keep_labels = unique_labels[counts >= min_cluster_size_for_metric]
        if len(keep_labels) < 2:
            return np.nan
        keep_mask = np.isin(labels_valid, keep_labels)
        labels_valid = labels_valid[keep_mask]
        emb_valid = emb_valid[keep_mask]

    unique_labels, counts = np.unique(labels_valid, return_counts=True)
    if len(unique_labels) < 2:
        return np.nan
    if np.any(counts < min_cluster_size_for_metric):
        return np.nan
    if len(labels_valid) < max(min_points_for_metric, len(unique_labels) + 1):
        return np.nan

    try:
        value = float(silhouette_score(emb_valid, labels_valid, metric=metric))
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan

def intra_inter_topic_similarity(
    embeddings,
    labels,
    outlier_label=-1,
    max_points_per_topic=SIMILARITY_MAX_POINTS_PER_TOPIC,
    random_state=42
):
    if embeddings is None or labels is None:
        return np.nan, np.nan

    labels_arr = np.asarray(labels)
    embeddings_arr = np.asarray(embeddings)

    if embeddings_arr.ndim != 2 or labels_arr.ndim != 1:
        return np.nan, np.nan
    if embeddings_arr.shape[0] != labels_arr.shape[0]:
        return np.nan, np.nan

    finite_mask = np.isfinite(labels_arr.astype(float, copy=False))
    valid_mask = (labels_arr != outlier_label) & finite_mask
    if valid_mask.sum() < 2:
        return np.nan, np.nan

    labels_valid = labels_arr[valid_mask]
    emb_valid = embeddings_arr[valid_mask]
    rng = np.random.default_rng(random_state)

    intra_values = []
    centroids = []

    for topic_label in np.unique(labels_valid):
        topic_mask = labels_valid == topic_label
        topic_emb = emb_valid[topic_mask]
        if topic_emb.shape[0] == 0:
            continue

        if topic_emb.shape[0] > max_points_per_topic:
            sampled_idx = rng.choice(topic_emb.shape[0], size=max_points_per_topic, replace=False)
            topic_emb = topic_emb[sampled_idx]

        centroids.append(topic_emb.mean(axis=0))

        if topic_emb.shape[0] >= 2:
            sim_matrix = cosine_similarity(topic_emb)
            upper_idx = np.triu_indices(sim_matrix.shape[0], k=1)
            if len(upper_idx[0]) > 0:
                intra_values.append(float(np.nanmean(sim_matrix[upper_idx])))

    intra_topic = float(np.nanmean(intra_values)) if len(intra_values) > 0 else np.nan

    if len(centroids) < 2:
        inter_topic = np.nan
    else:
        centroids_arr = np.vstack(centroids)
        centroid_sim = cosine_similarity(centroids_arr)
        upper_idx = np.triu_indices(centroid_sim.shape[0], k=1)
        inter_topic = float(np.nanmean(centroid_sim[upper_idx])) if len(upper_idx[0]) > 0 else np.nan

    return intra_topic, inter_topic

def coherence_score(topics, texts, dictionary, coherence_type='c_v', jitter=COHERENCE_EPS):
    clean_texts = [t for t in texts if isinstance(t, list) and len(t) >= 2]
    if len(clean_texts) < COHERENCE_MIN_TEXTS or len(dictionary) < COHERENCE_MIN_VOCAB:
        return np.nan if jitter <= 0 else float(jitter)

    local_dict = Dictionary(clean_texts)
    local_dict.filter_extremes(no_below=2, no_above=0.95)
    local_dict.compactify()
    if len(local_dict) < 20:
        return np.nan if jitter <= 0 else float(jitter)

    filtered_topics = _filter_topics_for_dictionary(topics, local_dict, min_words_per_topic=3)
    if len(filtered_topics) < 2:
        return np.nan if jitter <= 0 else float(jitter)

    def _compute_value(kind):
        if kind == 'u_mass':
            corpus = [local_dict.doc2bow(doc) for doc in clean_texts]
            cm = CoherenceModel(
                topics=filtered_topics,
                corpus=corpus,
                dictionary=local_dict,
                coherence='u_mass'
            )
        else:
            cm = CoherenceModel(
                topics=filtered_topics,
                texts=clean_texts,
                dictionary=local_dict,
                coherence=kind
            )
        return float(cm.get_coherence())

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning, module='gensim')
        with np.errstate(divide='ignore', invalid='ignore'):
            try:
                value = _compute_value(coherence_type)
                if np.isfinite(value):
                    return float(value)
            except Exception:
                pass

            # Only c_v gets u_mass fallback. For c_npmi we avoid mixed-scale fallback.
            if coherence_type == 'c_v':
                try:
                    fallback_value = _compute_value('u_mass')
                    if np.isfinite(fallback_value):
                        return float(fallback_value)
                except Exception:
                    pass

    return np.nan if jitter <= 0 else float(jitter)

cv_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_v')
npmi_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_npmi')
cv_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_v')
npmi_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_npmi')
div_train = topic_diversity(bertopic_topics)
val_outlier_ratio = float(np.mean(np.array(val_topics) == -1)) if len(val_topics) > 0 else np.nan
valid_topic_count = len(_filter_topics_for_dictionary(bertopic_topics, id2word, min_words_per_topic=3))

val_embeddings_for_metrics = sentence_model.encode(
    val_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
sil_val = safe_silhouette_score(val_embeddings_for_metrics, val_topics)
sil_val = 0.0 if not np.isfinite(sil_val) else sil_val
intra_val, inter_val = intra_inter_topic_similarity(val_embeddings_for_metrics, val_topics)

print(f"BERTopic Coherence C_v (train): {cv_train:.4f}")
print(f"BERTopic Coherence NPMI (train): {npmi_train:.4f}")
print(f"BERTopic Coherence C_v (val):   {cv_val:.4f}")
print(f"BERTopic Coherence NPMI (val):   {npmi_val:.4f}")
print(f"BERTopic Topic Diversity (train): {div_train:.4f}")
print(f"Validation outlier ratio (-1 topics): {val_outlier_ratio:.4f}")
print(f"Validation silhouette (cosine, no outliers): {sil_val:.4f}")
print(f"Validation intra-topic similarity: {intra_val:.4f}")
print(f"Validation inter-topic similarity: {inter_val:.4f}")
print(f"Valid topic count for coherence: {valid_topic_count}")

Batches: 100%|██████████| 19/19 [00:00<00:00, 37.96it/s]

BERTopic Coherence C_v (train): 0.5401
BERTopic Coherence NPMI (train): 0.1184
BERTopic Coherence C_v (val):   0.4519
BERTopic Coherence NPMI (val):   -0.1348
BERTopic Topic Diversity (train): 0.8353
Validation outlier ratio (-1 topics): 0.5477
Validation silhouette (cosine, no outliers): 0.1505
Validation intra-topic similarity: 0.5031
Validation inter-topic similarity: 0.3860
Valid topic count for coherence: 51


Hyperparameter tuning with rolling time-series CV (train+val only, fixed seed) and bucket-wise reporting

## Tuning Strategy & Composite Scoring Explained

This cell runs a **comprehensive hyperparameter search** with a Jehnen-style multi-metric ranking setup.

### Search Design
- **Space**: 300 targeted parameter combinations (n_neighbors × n_components × min_cluster_size × min_samples × ngram_range).
- **Folds**: 3 rolling/expanding time-series CV folds on train+val (test untouched).
- **Seed**: Fixed `RANDOM_SEED=42` for deterministic model selection.

### Per-Fold Metrics Used for Ranking (Equal Weighting)
| Metric | Direction | Notes |
|--------|-----------|-------|
| **c_v coherence** | Higher is better | Semantic coherence of topic words |
| **NPMI coherence** | Higher is better | Strong coherence signal for topic quality |
| **Topic diversity** | Higher is better | Fraction of unique words across topics |
| **Intra-topic similarity** | Higher is better | Cohesion within topic clusters in embedding space |
| **Inter-topic similarity** | Lower is better | Separation between topic clusters in embedding space |

### Aggregation & Normalization
1. Average each metric across folds per hyperparameter combination.
2. Min-max normalize each metric to [0,1].
3. Invert inter-topic similarity during normalization so lower raw inter-topic similarity gets higher normalized score.

### Composite Scoring Formula (No Manual Weighting)
All five ranking metrics are weighted equally:

$$
\text{score} = \frac{1}{5}\left(
\text{c\_v}_{\text{norm}} + \text{npmi}_{\text{norm}} + \text{diversity}_{\text{norm}} + \text{intra}_{\text{norm}} + \text{inter\_separation}_{\text{norm}}
\right)
$$

This is equivalent to **no preferential weighting** between ranking metrics.

### Model Selection
The highest composite score row is selected as `best_params`. Fold-level best rows are also reported to inspect temporal stability.

### Next: Multi-Seed Final Evaluation
After selecting best params, the final model is re-fit and evaluated across 9 random seeds on the untouched test set.

In [14]:
from itertools import product
import time

# ================================================================================
# HYPERPARAMETER TUNING: GRID SEARCH + ROLLING CV + JEHNEN-STYLE EQUAL-WEIGHT RANKING
# ================================================================================
# Ranking metrics (equal weight):
#   - c_v coherence
#   - c_npmi coherence
#   - topic diversity
#   - intra-topic similarity (higher better)
#   - inter-topic similarity (lower better; inverted during normalization)
# ================================================================================

# -------------------- Time-series CV controls (fixed seed for model selection) --------------------
N_FOLDS = 3  # rolling/expanding buckets before test
RANDOM_SEED = 42

# Build tuning pool from train + validation only (test remains untouched)
tune_df = pd.concat([train_df, val_df], axis=0).sort_values('date').reset_index(drop=True)
tune_docs = tune_df['headline'].tolist()
tune_timestamps = tune_df['date'].tolist()

# Precompute embeddings for the full tuning pool once
tune_embeddings = sentence_model.encode(
    tune_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

def sanitize_metric(value, fallback=np.nan):
    try:
        v = float(value)
    except Exception:
        return fallback
    return v if np.isfinite(v) else fallback

def make_expanding_folds(df_dates, n_folds=3, embargo_n=1):
    unique_days = np.array(sorted(pd.to_datetime(df_dates).dt.floor('D').unique()))
    n_days = len(unique_days)
    edges = np.linspace(0, n_days, n_folds + 2, dtype=int)
    day_code, _ = pd.factorize(pd.to_datetime(df_dates).dt.floor('D'), sort=True)

    folds = []
    for fold_idx in range(n_folds):
        train_end = edges[fold_idx + 1]
        val_start = train_end + embargo_n
        val_end = edges[fold_idx + 2]

        if train_end <= 0 or val_start >= val_end:
            continue

        train_mask = day_code < train_end
        val_mask = (day_code >= val_start) & (day_code < val_end)

        train_idx = np.where(train_mask)[0]
        val_idx = np.where(val_mask)[0]

        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        folds.append({
            'fold': fold_idx + 1,
            'train_idx': train_idx,
            'val_idx': val_idx,
            'train_start': unique_days[0],
            'train_end': unique_days[train_end - 1],
            'val_start': unique_days[val_start],
            'val_end': unique_days[val_end - 1]
        })

    return folds

folds = make_expanding_folds(tune_df['date'], n_folds=N_FOLDS)
print(f'Generated {len(folds)} rolling folds (train+val pool only).')
for fold in folds:
    print(
        f"Fold {fold['fold']}: train {fold['train_start']} -> {fold['train_end']} | "
        f"val {fold['val_start']} -> {fold['val_end']}"
    )

param_grid = {
    'n_neighbors': [10, 30, 50, 70, 90],
    'n_components': [4, 6, 8],
    'min_cluster_size': [8, 12, 16, 24, 32],
    'min_samples': [1, 2, 4, 6],
    'ngram_range': [(1, 2)]
}

search_space = list(product(
    param_grid['n_neighbors'],
    param_grid['n_components'],
    param_grid['min_cluster_size'],
    param_grid['min_samples'],
    param_grid['ngram_range']
))
expected_combinations = (
    len(param_grid['n_neighbors'])
    * len(param_grid['n_components'])
    * len(param_grid['min_cluster_size'])
    * len(param_grid['min_samples'])
    * len(param_grid['ngram_range'])
)

print(f'Configured combinations: {expected_combinations} (target ~300)')
print(f'Running {len(search_space)} parameter combinations x {len(folds)} folds (fixed seed={RANDOM_SEED})')

cv_rows = []

for trial, (n_neighbors, n_components, min_cluster_size, min_samples, ngram_range) in enumerate(search_space, start=1):
    print(
        f"[Trial {trial}/{len(search_space)}] n_neighbors={n_neighbors}, n_components={n_components}, "
        f"min_cluster_size={min_cluster_size}, min_samples={min_samples}, ngram_range={ngram_range}"
    )

    for fold in folds:
        train_idx = fold['train_idx']
        val_idx = fold['val_idx']

        fold_train_docs = [tune_docs[i] for i in train_idx]
        fold_val_docs = [tune_docs[i] for i in val_idx]
        fold_train_embeddings = tune_embeddings[train_idx]
        fold_val_embeddings = tune_embeddings[val_idx]

        processed_train_fold = preprocess_documents(fold_train_docs)
        processed_val_fold = preprocess_documents(fold_val_docs)
        id2word_fold = Dictionary(processed_train_fold)

        start = time.perf_counter()

        umap_model_i = UMAP(
            n_neighbors=n_neighbors,
            n_components=n_components,
            metric='cosine',
            random_state=RANDOM_SEED
        )
        hdbscan_model_i = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True,
            core_dist_n_jobs=-1
        )
        vectorizer_model_i = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=ngram_range)

        topic_model_i = BERTopic(
            embedding_model=sentence_model,
            umap_model=umap_model_i,
            hdbscan_model=hdbscan_model_i,
            vectorizer_model=vectorizer_model_i,
            calculate_probabilities=False,
            verbose=False
        )

        _topics_train_i, _ = topic_model_i.fit_transform(fold_train_docs, embeddings=fold_train_embeddings)
        val_topics_i, _ = topic_model_i.transform(fold_val_docs, embeddings=fold_val_embeddings)

        topic_words_i = [
            [word for word, _ in topic_model_i.get_topic(t)]
            for t in topic_model_i.get_topics().keys()
            if t != -1 and topic_model_i.get_topic(t) is not None
        ]

        has_enough_for_coherence = (
            len(processed_val_fold) >= 20 and
            len(id2word_fold) >= 30 and
            len(topic_words_i) >= 2
        )

        cv_val_i = sanitize_metric(
            coherence_score(topic_words_i, processed_val_fold, id2word_fold, 'c_v', jitter=COHERENCE_EPS),
            fallback=COHERENCE_EPS
        ) if has_enough_for_coherence else COHERENCE_EPS

        npmi_val_i = sanitize_metric(
            coherence_score(topic_words_i, processed_val_fold, id2word_fold, 'c_npmi', jitter=COHERENCE_EPS),
            fallback=COHERENCE_EPS
        ) if has_enough_for_coherence else COHERENCE_EPS

        div_i = sanitize_metric(topic_diversity(topic_words_i), fallback=0.0)
        intra_i, inter_i = intra_inter_topic_similarity(fold_val_embeddings, val_topics_i)
        intra_i = sanitize_metric(intra_i, fallback=0.0)
        inter_i = sanitize_metric(inter_i, fallback=1.0)

        n_topics_i = int(sum(1 for t in topic_model_i.get_topics().keys() if int(t) != -1))
        elapsed = time.perf_counter() - start

        cv_rows.append({
            'trial': trial,
            'fold': fold['fold'],
            'random_seed': RANDOM_SEED,
            'n_neighbors': n_neighbors,
            'n_components': n_components,
            'min_cluster_size': min_cluster_size,
            'min_samples': min_samples,
            'ngram_range': str(ngram_range),
            'n_topics': n_topics_i,
            'cv_val': cv_val_i,
            'npmi_val': npmi_val_i,
            'topic_diversity': div_i,
            'intra_topic_similarity': intra_i,
            'inter_topic_similarity': inter_i,
            'fit_seconds': elapsed
        })

cv_results = pd.DataFrame(cv_rows)

agg_cols = [
    'cv_val', 'npmi_val', 'topic_diversity',
    'intra_topic_similarity', 'inter_topic_similarity',
    'n_topics', 'fit_seconds'
]
tuning_results = cv_results.groupby(
    ['n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
)[agg_cols].mean()

def minmax_norm(series, higher_is_better=True):
    s = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    if s.dropna().empty:
        return pd.Series(0.0, index=s.index)
    filled = s.fillna(s.min() if higher_is_better else s.max())
    min_v, max_v = float(filled.min()), float(filled.max())
    if max_v == min_v:
        return pd.Series(0.0, index=filled.index)
    norm = (filled - min_v) / (max_v - min_v)
    return norm

tuning_results['cv_val_norm'] = minmax_norm(tuning_results['cv_val'], higher_is_better=True)
tuning_results['npmi_val_norm'] = minmax_norm(tuning_results['npmi_val'], higher_is_better=True)
tuning_results['topic_diversity_norm'] = minmax_norm(tuning_results['topic_diversity'], higher_is_better=True)
tuning_results['intra_topic_similarity_norm'] = minmax_norm(tuning_results['intra_topic_similarity'], higher_is_better=True)
tuning_results['inter_topic_separation_norm'] = minmax_norm(tuning_results['inter_topic_similarity'], higher_is_better=False)

# Equal weighting across Jehnen-style ranking metrics (no preferential weighting)
tuning_results['composite_score'] = (
    tuning_results['cv_val_norm'] +
    tuning_results['npmi_val_norm'] +
    tuning_results['topic_diversity_norm'] +
    tuning_results['intra_topic_similarity_norm'] +
    tuning_results['inter_topic_separation_norm']
) / 5.0

tuning_results = tuning_results.sort_values('composite_score', ascending=False).reset_index(drop=True)
best_params = tuning_results.iloc[0].to_dict()

# Bucket-wise (fold-wise) best params to inspect drift over time
fold_agg = cv_results.groupby(
    ['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
)[['cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity', 'n_topics']].mean()

fold_agg['cv_val_norm'] = fold_agg.groupby('fold')['cv_val'].transform(lambda x: minmax_norm(x, True))
fold_agg['npmi_val_norm'] = fold_agg.groupby('fold')['npmi_val'].transform(lambda x: minmax_norm(x, True))
fold_agg['topic_diversity_norm'] = fold_agg.groupby('fold')['topic_diversity'].transform(lambda x: minmax_norm(x, True))
fold_agg['intra_topic_similarity_norm'] = fold_agg.groupby('fold')['intra_topic_similarity'].transform(lambda x: minmax_norm(x, True))
fold_agg['inter_topic_separation_norm'] = fold_agg.groupby('fold')['inter_topic_similarity'].transform(lambda x: minmax_norm(x, False))

fold_agg['fold_score'] = (
    fold_agg['cv_val_norm'] +
    fold_agg['npmi_val_norm'] +
    fold_agg['topic_diversity_norm'] +
    fold_agg['intra_topic_similarity_norm'] +
    fold_agg['inter_topic_separation_norm']
) / 5.0

best_per_fold = (
    fold_agg.sort_values(['fold', 'fold_score'], ascending=[True, False])
            .groupby('fold', as_index=False)
            .head(1)
            .reset_index(drop=True)
)

print('\nTop configurations (overall):')
display(tuning_results.head(10))

print('\nBest params by time bucket (fold):')
display(best_per_fold[['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range', 'cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity', 'fold_score']])

print('\nAverage fit time per model run (seconds):', round(float(cv_results['fit_seconds'].mean()), 2))
print('Best overall configuration selected from rolling CV (fixed-seed model selection):')
display(pd.DataFrame([best_params]))

Batches: 100%|██████████| 50/50 [00:00<00:00, 85.37it/s]


Generated 3 rolling folds (train+val pool only).
Fold 1: train 2024-09-17 00:00:00 -> 2025-07-03 00:00:00 | val 2025-07-05 00:00:00 -> 2025-09-24 00:00:00
Fold 2: train 2024-09-17 00:00:00 -> 2025-09-24 00:00:00 | val 2025-09-26 00:00:00 -> 2025-12-15 00:00:00
Fold 3: train 2024-09-17 00:00:00 -> 2025-12-15 00:00:00 | val 2025-12-17 00:00:00 -> 2026-03-10 00:00:00
Configured combinations: 300 (target ~300)
Running 300 parameter combinations x 3 folds (fixed seed=42)
[Trial 1/300] n_neighbors=10, n_components=4, min_cluster_size=8, min_samples=1, ngram_range=(1, 2)
[Trial 2/300] n_neighbors=10, n_components=4, min_cluster_size=8, min_samples=2, ngram_range=(1, 2)
[Trial 3/300] n_neighbors=10, n_components=4, min_cluster_size=8, min_samples=4, ngram_range=(1, 2)
[Trial 4/300] n_neighbors=10, n_components=4, min_cluster_size=8, min_samples=6, ngram_range=(1, 2)
[Trial 5/300] n_neighbors=10, n_components=4, min_cluster_size=12, min_samples=1, ngram_range=(1, 2)
[Trial 6/300] n_neighbors=10

,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,n_topics,fit_seconds,cv_val_norm,npmi_val_norm,topic_diversity_norm,intra_topic_similarity_norm,inter_topic_separation_norm,composite_score
0,10,6,8,6,"(1, 2)",0.604067,0.055760,0.931209,0.493265,0.425934,40.333333,17.502363,1.000000,1.000000,0.584891,0.504636,0.419372,0.701780
1,90,6,32,6,"(1, 2)",0.532375,0.029036,0.958333,0.401298,0.463642,3.000000,10.250959,0.672129,0.897527,0.859426,0.258813,0.503948,0.638369
2,50,4,12,6,"(1, 2)",0.535592,0.015578,0.947340,0.435585,0.473496,17.666667,18.000883,0.686841,0.845921,0.748163,0.350462,0.526052,0.631488
3,50,4,16,6,"(1, 2)",0.532710,0.011236,0.950676,0.427563,0.478042,14.333333,17.947492,0.673659,0.829271,0.781920,0.329018,0.536249,0.630023
4,30,6,32,6,"(1, 2)",0.495493,0.004121,0.955556,0.371783,0.573921,3.333333,9.725361,0.503455,0.801990,0.831311,0.179922,0.751301,0.613596
5,30,6,24,6,"(1, 2)",0.492036,0.011975,0.953030,0.386036,0.559379,5.000000,9.979267,0.487643,0.832104,0.805752,0.218018,0.718683,0.612440
6,90,4,8,6,"(1, 2)",0.485512,-0.064339,0.921916,0.621502,0.535152,34.333333,18.046186,0.457808,0.539478,0.490833,0.847407,0.664344,0.599974
7,90,6,24,2,"(1, 2)",0.533020,-0.009220,0.946667,0.390787,0.507060,3.333333,10.229616,0.675078,0.750833,0.741343,0.230718,0.601333,0.599861
8,90,6,32,2,"(1, 2)",0.533020,-0.009220,0.946667,0.390787,0.507060,3.333333,10.208655,0.675078,0.750833,0.741343,0.230718,0.601333,0.599861
9,30,4,16,6,"(1, 2)",0.534343,-0.002502,0.951852,0.445740,0.401089,15.666667,17.442917,0.681126,0.776593,0.793824,0.377604,0.363644,0.598558



Best params by time bucket (fold):


,fold,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,fold_score
0,1,10,6,8,6,"(1, 2)",0.668795,0.130341,1.0,0.477042,0.500933,0.773177
1,2,30,4,16,6,"(1, 2)",0.678505,0.199068,1.0,0.450767,0.401826,0.759435
2,3,90,6,24,1,"(1, 2)",0.618626,0.207390,1.0,0.406386,0.511126,0.701330



Average fit time per model run (seconds): 15.95
Best overall configuration selected from rolling CV (fixed-seed model selection):


,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,n_topics,fit_seconds,cv_val_norm,npmi_val_norm,topic_diversity_norm,intra_topic_similarity_norm,inter_topic_separation_norm,composite_score
0,10,6,8,6,"(1, 2)",0.604067,0.05576,0.931209,0.493265,0.425934,40.333333,17.502363,1.0,1.0,0.584891,0.504636,0.419372,0.70178


Final model and test evaluation with multi-seed variability analysis (single-touch per seed on test split)

In [15]:
from ast import literal_eval

# Refit best hyperparameter configuration on train+val, then evaluate variability across random seeds on test
EVAL_SEEDS = [42, 7, 123, 2024, 99, 11, 77, 314, 2718]

train_val_docs = train_docs + val_docs
train_val_embeddings = sentence_model.encode(
    train_val_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
test_embeddings = sentence_model.encode(
    test_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

processed_train_val_docs = preprocess_documents(train_val_docs)
processed_test_docs = preprocess_documents(test_docs)
id2word_train_val = Dictionary(processed_train_val_docs)

best_ngram = literal_eval(best_params['ngram_range'])
seed_rows = []

for seed in EVAL_SEEDS:
    print(f'Running final fit/eval for seed={seed}...')

    best_umap = UMAP(
        n_neighbors=int(best_params['n_neighbors']),
        n_components=int(best_params['n_components']),
        metric='cosine',
        random_state=seed
    )
    best_hdbscan = HDBSCAN(
        min_cluster_size=int(best_params['min_cluster_size']),
        min_samples=int(best_params['min_samples']),
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    best_vectorizer = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=best_ngram)

    best_topic_model = BERTopic(
        embedding_model=sentence_model,
        umap_model=best_umap,
        hdbscan_model=best_hdbscan,
        vectorizer_model=best_vectorizer,
        calculate_probabilities=True,
        verbose=False
    )

    _topics_train_val, _ = best_topic_model.fit_transform(train_val_docs, embeddings=train_val_embeddings)
    test_topics, _ = best_topic_model.transform(test_docs, embeddings=test_embeddings)

    best_topic_words = [
        [word for word, _ in best_topic_model.get_topic(t)]
        for t in best_topic_model.get_topics().keys()
        if t != -1 and best_topic_model.get_topic(t) is not None
    ]

    cv_test = sanitize_metric(
        coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_v', jitter=COHERENCE_EPS),
        fallback=COHERENCE_EPS
    )
    npmi_test = sanitize_metric(
        coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_npmi', jitter=COHERENCE_EPS),
        fallback=COHERENCE_EPS
    )
    div_test = sanitize_metric(topic_diversity(best_topic_words), fallback=0.0)
    intra_test, inter_test = intra_inter_topic_similarity(test_embeddings, test_topics)
    intra_test = sanitize_metric(intra_test, fallback=0.0)
    inter_test = sanitize_metric(inter_test, fallback=1.0)

    # Final-only diagnostics (not used in tuning)
    sil_test = sanitize_metric(
        safe_silhouette_score(test_embeddings, test_topics, outlier_label=-1, metric='cosine'),
        fallback=0.0
    )
    test_outlier_ratio = float(np.mean(np.array(test_topics) == -1)) if len(test_topics) > 0 else np.nan
    test_topic_count = int(sum(1 for t in best_topic_model.get_topics().keys() if int(t) != -1))

    seed_rows.append({
        'seed': seed,
        'cv_test': cv_test,
        'npmi_test': npmi_test,
        'topic_diversity_test': div_test,
        'intra_topic_similarity_test': intra_test,
        'inter_topic_similarity_test': inter_test,
        'test_silhouette': sil_test,
        'test_outlier_ratio': test_outlier_ratio,
        'test_topic_count': test_topic_count
    })

seed_results = pd.DataFrame(seed_rows)

print('\nPer-seed test metrics:')
display(seed_results)

summary_stats = pd.DataFrame([{
    'cv_test_mean': seed_results['cv_test'].mean(skipna=True),
    'cv_test_std': seed_results['cv_test'].std(skipna=True),
    'npmi_test_mean': seed_results['npmi_test'].mean(skipna=True),
    'npmi_test_std': seed_results['npmi_test'].std(skipna=True),
    'topic_diversity_mean': seed_results['topic_diversity_test'].mean(skipna=True),
    'topic_diversity_std': seed_results['topic_diversity_test'].std(skipna=True),
    'intra_topic_similarity_mean': seed_results['intra_topic_similarity_test'].mean(skipna=True),
    'intra_topic_similarity_std': seed_results['intra_topic_similarity_test'].std(skipna=True),
    'inter_topic_similarity_mean': seed_results['inter_topic_similarity_test'].mean(skipna=True),
    'inter_topic_similarity_std': seed_results['inter_topic_similarity_test'].std(skipna=True),
    'test_silhouette_mean': seed_results['test_silhouette'].mean(skipna=True),
    'test_silhouette_std': seed_results['test_silhouette'].std(skipna=True),
    'test_outlier_ratio_mean': seed_results['test_outlier_ratio'].mean(skipna=True),
    'test_outlier_ratio_std': seed_results['test_outlier_ratio'].std(skipna=True),
    'test_topic_count_mean': seed_results['test_topic_count'].mean(skipna=True),
    'test_topic_count_std': seed_results['test_topic_count'].std(skipna=True)
}])

print('\nFinal test metrics variability across random seeds:')
display(summary_stats)

def format_mean_std(mean_value, std_value, digits=4):
    if not np.isfinite(mean_value):
        return 'NA'
    if not np.isfinite(std_value):
        return f"{mean_value:.{digits}f} ± NA"
    return f"{mean_value:.{digits}f} ± {std_value:.{digits}f}"

thesis_report = pd.DataFrame([{
    'Model': 'BERTopic (best CV config)',
    'Seeds (n)': len(EVAL_SEEDS),
    'Coherence C_v (test)': format_mean_std(summary_stats.at[0, 'cv_test_mean'], summary_stats.at[0, 'cv_test_std'], digits=4),
    'Coherence NPMI (test)': format_mean_std(summary_stats.at[0, 'npmi_test_mean'], summary_stats.at[0, 'npmi_test_std'], digits=4),
    'Topic Diversity (test)': format_mean_std(summary_stats.at[0, 'topic_diversity_mean'], summary_stats.at[0, 'topic_diversity_std'], digits=4),
    'Intra-topic Similarity (test)': format_mean_std(summary_stats.at[0, 'intra_topic_similarity_mean'], summary_stats.at[0, 'intra_topic_similarity_std'], digits=4),
    'Inter-topic Similarity (test)': format_mean_std(summary_stats.at[0, 'inter_topic_similarity_mean'], summary_stats.at[0, 'inter_topic_similarity_std'], digits=4),
    'Silhouette (test, cosine)': format_mean_std(summary_stats.at[0, 'test_silhouette_mean'], summary_stats.at[0, 'test_silhouette_std'], digits=4),
    'Outlier Ratio (test)': format_mean_std(summary_stats.at[0, 'test_outlier_ratio_mean'], summary_stats.at[0, 'test_outlier_ratio_std'], digits=4),
    'Topic Count (test)': format_mean_std(summary_stats.at[0, 'test_topic_count_mean'], summary_stats.at[0, 'test_topic_count_std'], digits=2)
}])

print('\nThesis-ready reporting table (mean ± std across seeds):')
display(thesis_report)

Batches: 100%|██████████| 18/18 [00:00<00:00, 126.64it/s]


Running final fit/eval for seed=42...
Running final fit/eval for seed=7...
Running final fit/eval for seed=123...
Running final fit/eval for seed=2024...
Running final fit/eval for seed=99...
Running final fit/eval for seed=11...
Running final fit/eval for seed=77...
Running final fit/eval for seed=314...
Running final fit/eval for seed=2718...

Per-seed test metrics:


,seed,cv_test,npmi_test,topic_diversity_test,intra_topic_similarity_test,inter_topic_similarity_test,test_silhouette,test_outlier_ratio,test_topic_count
0,42,0.471286,-0.145256,0.873451,0.463310,0.334473,0.063526,0.484375,113
1,7,0.464392,-0.132137,0.878431,0.476188,0.334643,0.067793,0.474826,102
2,123,0.469866,-0.120976,0.864706,0.462485,0.357436,0.071016,0.457465,102
3,2024,0.445390,-0.151330,0.875229,0.459833,0.337192,0.067100,0.458333,109
4,99,0.466790,-0.126473,0.874312,0.468642,0.352694,0.058763,0.447917,109
5,11,0.483483,-0.105467,0.885849,0.455718,0.325681,0.061262,0.483507,106
6,77,0.455564,-0.158956,0.870093,0.481169,0.332664,0.060856,0.470486,107
7,314,0.465042,-0.147739,0.883636,0.473672,0.336406,0.068785,0.475694,110
8,2718,0.478925,-0.116523,0.875758,0.460823,0.336370,0.060744,0.445312,99



Final test metrics variability across random seeds:


,cv_test_mean,cv_test_std,npmi_test_mean,npmi_test_std,topic_diversity_mean,topic_diversity_std,intra_topic_similarity_mean,intra_topic_similarity_std,inter_topic_similarity_mean,inter_topic_similarity_std,test_silhouette_mean,test_silhouette_std,test_outlier_ratio_mean,test_outlier_ratio_std,test_topic_count_mean,test_topic_count_std
0,0.466749,0.011447,-0.133873,0.017985,0.875718,0.006451,0.466871,0.008537,0.338618,0.01,0.064427,0.004332,0.466435,0.014667,106.333333,4.527693



Thesis-ready reporting table (mean ± std across seeds):


,Model,Seeds (n),Coherence C_v (test),Coherence NPMI (test),Topic Diversity (test),Intra-topic Similarity (test),Inter-topic Similarity (test),"Silhouette (test, cosine)",Outlier Ratio (test),Topic Count (test)
0,BERTopic (best CV config),9,0.4667 ± 0.0114,-0.1339 ± 0.0180,0.8757 ± 0.0065,0.4669 ± 0.0085,0.3386 ± 0.0100,0.0644 ± 0.0043,0.4664 ± 0.0147,106.33 ± 4.53


In [16]:
import plotly.express as px

# Diagnostics use the currently available final test assignment (`test_topics`) and embeddings (`test_embeddings`).
if 'test_topics' not in globals() or 'test_embeddings' not in globals():
    raise RuntimeError('Please run Cell 24 first so `test_topics` and `test_embeddings` are available.')

labels = np.asarray(test_topics)
emb = np.asarray(test_embeddings)

if labels.shape[0] != emb.shape[0]:
    raise ValueError(f'Mismatch: labels={labels.shape[0]} vs embeddings={emb.shape[0]}')

is_outlier = labels == -1
outlier_ratio_now = float(np.mean(is_outlier)) if len(labels) > 0 else np.nan

valid_labels = labels[~is_outlier]
if len(valid_labels) > 0:
    cluster_size_series = pd.Series(valid_labels).value_counts().sort_values(ascending=False)
else:
    cluster_size_series = pd.Series(dtype='int64')

n_valid_clusters_now = int(cluster_size_series.shape[0])
singleton_topics_now = set(cluster_size_series[cluster_size_series == 1].index.tolist())
n_singletons_now = int(len(singleton_topics_now))

print('Final-clustering diagnostics (current test assignment):')
print(f'  Documents: {len(labels)}')
print(f'  Outlier ratio (-1): {outlier_ratio_now:.4f}')
print(f'  Valid clusters (excluding -1): {n_valid_clusters_now}')
print(f'  Singleton clusters: {n_singletons_now}')

if n_valid_clusters_now == 0:
    print('⚠️ No valid clusters found (all points are outliers).')
elif n_valid_clusters_now == 1:
    print('⚠️ Only one valid cluster found. Silhouette is undefined by design.')
if n_singletons_now > 0:
    print('⚠️ Singleton clusters present. Silhouette excludes them in our robust function.')

# 1) Cluster size distribution (excluding outliers)
if n_valid_clusters_now > 0:
    cluster_size_df = (
        cluster_size_series
        .rename_axis('Topic')
        .reset_index(name='Count')
        .sort_values('Count', ascending=False)
    )
    cluster_size_df['Topic'] = cluster_size_df['Topic'].astype(int)

    fig_sizes = px.bar(
        cluster_size_df,
        x='Topic',
        y='Count',
        title='Final Test Cluster Sizes (excluding outliers)',
        labels={'Topic': 'Topic ID', 'Count': 'Number of Documents'}
    )
    fig_sizes.show()
else:
    print('Cluster-size plot skipped (no valid clusters).')

# 2) 2D embedding view colored by diagnostic point type
# Recompute a 2D projection from test embeddings for visual inspection.
umap_viz = UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
emb_2d = umap_viz.fit_transform(emb)

point_type = np.where(
    labels == -1,
    'Outlier (-1)',
    np.where(np.isin(labels, list(singleton_topics_now)), 'Singleton cluster', 'Valid cluster')
)

viz_df = pd.DataFrame({
    'x': emb_2d[:, 0],
    'y': emb_2d[:, 1],
    'topic': labels.astype(int),
    'point_type': point_type
})

fig_diag = px.scatter(
    viz_df,
    x='x',
    y='y',
    color='point_type',
    hover_data=['topic'],
    title='Final Test Embedding Diagnostics (Outliers / Singleton / Valid)'
)
fig_diag.update_traces(marker=dict(size=6, opacity=0.7))
fig_diag.show()

# 3) Seed-level context from the multi-seed run
if 'seed_results' in globals() and isinstance(seed_results, pd.DataFrame) and not seed_results.empty:
    seed_plot_df = seed_results.copy()

    fig_seed_outliers = px.bar(
        seed_plot_df,
        x='seed',
        y='test_outlier_ratio',
        title='Outlier Ratio by Seed',
        labels={'seed': 'Seed', 'test_outlier_ratio': 'Outlier Ratio'}
    )
    fig_seed_outliers.show()

    fig_seed_topics = px.bar(
        seed_plot_df,
        x='seed',
        y='test_topic_count',
        title='Valid Topic Count by Seed',
        labels={'seed': 'Seed', 'test_topic_count': 'Topic Count (excluding -1)'}
    )
    fig_seed_topics.show()
else:
    print('Seed-level plots skipped (run Cell 24 first to populate `seed_results`).')

Final-clustering diagnostics (current test assignment):
  Documents: 1152
  Outlier ratio (-1): 0.4453
  Valid clusters (excluding -1): 82
  Singleton clusters: 20
⚠️ Singleton clusters present. Silhouette excludes them in our robust function.


Cluster diagnostics and visual sanity checks (outliers, valid clusters, singleton clusters)

## Export Results

Save all outputs to timestamped folder for reuse in other files and scientific results.

In [17]:
from pathlib import Path
from datetime import datetime
import json

# Validate required variables
try:
    assert 'df' in locals(), "df not found in workspace"
    assert 'train_df' in locals(), "train_df not found in workspace"
    assert 'val_df' in locals(), "val_df not found in workspace"
    assert 'test_df' in locals(), "test_df not found in workspace"
    assert 'topic_model' in locals(), "topic_model not found in workspace"
    print("✓ All required variables found")
except AssertionError as e:
    raise RuntimeError(f"Export validation failed: {e}")

# Create timestamped output directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_base = Path("Outputs/Finnhub_BERTopic")
output_dir = output_base / timestamp
output_dir.mkdir(parents=True, exist_ok=True)

# Create subdirectories
(output_dir / "tables").mkdir(exist_ok=True)
(output_dir / "row_level").mkdir(exist_ok=True)
(output_dir / "models").mkdir(exist_ok=True)
(output_dir / "figures").mkdir(exist_ok=True)
(output_dir / "meta").mkdir(exist_ok=True)

print(f"\n📁 Output directory: {output_dir}")

# Helper function to save DataFrames
def save_df(df, filename, subfolder="tables"):
    filepath = output_dir / subfolder / f"{filename}.csv"
    df.to_csv(filepath, index=False, encoding='utf-8-sig')
    print(f"  ✓ Saved: {subfolder}/{filename}.csv")
    return filepath

# 1. Topic Information
topic_info = topic_model.get_topic_info()
save_df(topic_info, "topic_info")

# 2. Topic Terms (long format)
topic_terms_list = []
for topic_id in topic_model.get_topics().keys():
    if topic_id != -1:
        terms = topic_model.get_topic(topic_id)
        for word, weight in terms:
            topic_terms_list.append({"topic": topic_id, "word": word, "weight": weight})
topic_terms_long = pd.DataFrame(topic_terms_list)
save_df(topic_terms_long, "topic_terms_long")

# 3. Row-level topic assignments
try:
    row_topic_assignments = pd.DataFrame({
        'document_id': range(len(topic_model.topics_)),
        'topic': topic_model.topics_,
        'probabilities': [probs_train[i] if i < len(probs_train) else None for i in range(len(topic_model.topics_))]
    })
    save_df(row_topic_assignments, "row_topic_assignments_train", "row_level")
    print("  ✓ Saved: row_level/row_topic_assignments_train.csv")
except Exception as e:
    print(f"  ⚠ Could not save row-level assignments: {e}")

# 4. Save model artifact
try:
    model_path = output_dir / "models" / "bertopic_model"
    topic_model.save(str(model_path))
    print(f"  ✓ Saved: models/bertopic_model (BERTopic model)")
except Exception as e:
    print(f"  ⚠ Could not save model: {e}")

# 5. Summary metrics
summary_stats = {
    "dataset": "finnhub",
    "total_documents": len(df),
    "train_documents": len(train_df),
    "val_documents": len(val_df),
    "test_documents": len(test_df),
    "num_topics": topic_model.get_topic_info().shape[0] - 1,  # exclude -1 (outliers)
    "model_params": {
        "language": "english",
        "min_topic_size": topic_model.min_topic_size if hasattr(topic_model, 'min_topic_size') else "N/A",
    }
}

# 6. Export Plotly figures (optional - if they exist in the workspace)
saved_figures = []
figure_candidates = ['fig_daily', 'fig_topic_prev', 'fig_tradeoff', 'fig_diag']
for fig_name in figure_candidates:
    try:
        if fig_name in locals():
            fig_obj = locals()[fig_name]
            fig_path = output_dir / "figures" / f"{fig_name}.html"
            fig_obj.write_html(str(fig_path))
            saved_figures.append(fig_name)
            print(f"  ✓ Saved: figures/{fig_name}.html")
    except Exception as e:
        pass  # Silently skip if figure doesn't exist or can't be saved

# 7. Create and save export summary
export_summary = {
    "timestamp": timestamp,
    "dataset": "finnhub",
    "exported_items": {
        "tables": ["topic_info", "topic_terms_long"],
        "row_level": ["row_topic_assignments_train"],
        "models": ["bertopic_model"],
        "figures": saved_figures,
        "summary_stats": summary_stats
    },
    "output_directory": str(output_dir)
}

summary_path = output_dir / "meta" / "export_summary.json"
with open(summary_path, 'w') as f:
    json.dump(export_summary, f, indent=2, default=str)
print(f"  ✓ Saved: meta/export_summary.json")

print(f"\n✅ Export completed successfully!")
print(f"📊 Exported items: {', '.join(export_summary['exported_items']['tables'] + export_summary['exported_items']['row_level'] + export_summary['exported_items']['models'] + export_summary['exported_items']['figures'])}")
print(f"📁 Location: {output_dir}")

✓ All required variables found

📁 Output directory: Outputs\Finnhub_BERTopic\20260423_232228
  ✓ Saved: tables/topic_info.csv
  ✓ Saved: tables/topic_terms_long.csv


2026-04-23 23:22:28,502 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


  ✓ Saved: row_level/row_topic_assignments_train.csv
  ✓ Saved: row_level/row_topic_assignments_train.csv
  ✓ Saved: models/bertopic_model (BERTopic model)
  ✓ Saved: figures/fig_diag.html
  ✓ Saved: meta/export_summary.json

✅ Export completed successfully!
📊 Exported items: topic_info, topic_terms_long, row_topic_assignments_train, bertopic_model, fig_diag
📁 Location: Outputs\Finnhub_BERTopic\20260423_232228
